In [2]:
import cv2
import torch
import torchvision
import torchvision.transforms.functional as TF
import numpy as np
from PIL import Image
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import nms

# ──────────────────────────────────────────────────────────────
# ⚠️  CHANGE THESE IF NEEDED
# ──────────────────────────────────────────────────────────────
MODEL_PATH   = "/Users/taingmuyleang/Downloads/helmet_models/helmet_model_best.pth"
CAMERA_INDEX = 0
THRESHOLD    = 0.5   # lower = detect more people
NMS_IOU      = 0.3   # lower = stricter duplicate removal
# ──────────────────────────────────────────────────────────────

CLASSES     = ["__background__", "helmet", "no_helmet"]
BOX_COLORS  = {"helmet": (0, 255, 0), "no_helmet": (0, 0, 255)}
NUM_CLASSES = 3

# ── Build & load model ───────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print(f"✅ Model loaded  → {MODEL_PATH}")
print(f"   Running on   → {device}")


def predict_frame(frame):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    tensor  = TF.to_tensor(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)[0]

    boxes  = output["boxes"].cpu()
    labels = output["labels"].cpu()
    scores = output["scores"].cpu()

    # ── Step 1: Confidence filter ────────────────────────────
    keep   = scores >= THRESHOLD
    boxes  = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    if len(boxes) == 0:
        return frame

    # ── Step 2: Per-class NMS ─────────────────────────────────
    # Run NMS separately per class so multiple people get their own box
    final_indices = []
    for cls_id in [1, 2]:  # 1=helmet, 2=no_helmet
        cls_mask = labels == cls_id
        if cls_mask.sum() == 0:
            continue
        cls_boxes  = boxes[cls_mask]
        cls_scores = scores[cls_mask]
        cls_keep   = nms(cls_boxes, cls_scores, iou_threshold=NMS_IOU)
        # Map back to original indices
        original_indices = cls_mask.nonzero(as_tuple=True)[0]
        final_indices.extend(original_indices[cls_keep].tolist())

    if len(final_indices) == 0:
        return frame

    boxes  = boxes[final_indices]
    labels = labels[final_indices]
    scores = scores[final_indices]

    # ── Step 3: Cross-class duplicate removal ────────────────
    # If helmet and no_helmet box heavily overlap → keep higher score only
    final_keep = []
    suppressed = set()
    for i in range(len(boxes)):
        if i in suppressed:
            continue
        final_keep.append(i)
        for j in range(i + 1, len(boxes)):
            if j in suppressed:
                continue
            b1, b2   = boxes[i], boxes[j]
            inter_x1 = max(b1[0], b2[0])
            inter_y1 = max(b1[1], b2[1])
            inter_x2 = min(b1[2], b2[2])
            inter_y2 = min(b1[3], b2[3])
            inter_w  = max(0, inter_x2 - inter_x1)
            inter_h  = max(0, inter_y2 - inter_y1)
            inter    = inter_w * inter_h
            area1    = (b1[2] - b1[0]) * (b1[3] - b1[1])
            area2    = (b2[2] - b2[0]) * (b2[3] - b2[1])
            union    = area1 + area2 - inter
            iou      = inter / (union + 1e-6)
            if iou > 0.4:
                suppressed.add(j)

    boxes  = boxes[final_keep]
    labels = labels[final_keep]
    scores = scores[final_keep]

    # ── Step 4: Draw all boxes ───────────────────────────────
    helmet_count    = 0
    no_helmet_count = 0

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, box.numpy())
        name  = CLASSES[label.item()] if label.item() < len(CLASSES) else "unknown"
        color = BOX_COLORS.get(name, (0, 255, 255))

        if name == "helmet":
            helmet_count += 1
        elif name == "no_helmet":
            no_helmet_count += 1

        # Bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # Label background + text
        label_text = f"{name}: {score:.2f}"
        (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
        cv2.putText(frame, label_text, (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)

    # ── Step 5: Counter overlay ──────────────────────────────
    overlay = [
        f"Helmet    : {helmet_count}",
        f"No Helmet : {no_helmet_count}",
        f"People    : {helmet_count + no_helmet_count}",
        f"Threshold : {THRESHOLD}",
        f"Device    : {str(device).upper()}",
    ]
    for i, text in enumerate(overlay):
        y_pos = 30 + i * 30
        cv2.rectangle(frame, (8, y_pos - 22), (260, y_pos + 7), (0, 0, 0), -1)
        cv2.putText(frame, text, (12, y_pos),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

    return frame


# ── Real-time camera loop ────────────────────────────────────
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"❌ Cannot open camera at index {CAMERA_INDEX}.")
    print("   Try changing CAMERA_INDEX to 1 or 2 at the top.")
else:
    print(f"✅ Camera opened at index {CAMERA_INDEX}")
    print("   Press Q to quit.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("❌ Failed to grab frame.")
            break

        annotated = predict_frame(frame)
        cv2.imshow("Helmet Detection  |  Press Q to quit", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("🛑 Stopped by user.")
            break

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Camera released.")

✅ Model loaded  → /Users/taingmuyleang/Downloads/helmet_models/helmet_model_best.pth
   Running on   → cpu
✅ Camera opened at index 0
   Press Q to quit.


KeyboardInterrupt: 